In [3]:
from google.cloud.spark_connect import GoogleSparkSession
from google.cloud.dataproc_v1 import Session

session = Session()
# This is the first pull. If your default network is a legacy network, uncomment the next line, and use a non-legacy network
# session.environment_config.execution_config.subnetwork_uri = "dataproc-ci-tests"
# For using scheduling, specify a service account in your project below, and uncomment the line
# session.environment_config.execution_config.service_account = "359641935755-compute@developer.gserviceaccount.com"

spark = (
    GoogleSparkSession.builder
      .appName("CustomSparkSession")
      .googleSessionConfig(session)
      .getOrCreate()
)

Creating Spark session. It may take few minutes.
Interactive Session Detail View:  https://console.cloud.google.com/dataproc/interactive/us-central1/sc-20250502-022639-aad6o7?project=bsampath-da-2023


RuntimeError: Error while creating serverless session

In [2]:
# Load data from BigQuery
products = spark.read.format('bigquery') \
  .option('table', 'bigquery-public-data.thelook_ecommerce.products') \
  .load()
products.createOrReplaceTempView('products')

NameError: name 'spark' is not defined

In [ ]:
order_items = spark.read.format('bigquery') \
  .option('table', 'bigquery-public-data.thelook_ecommerce.order_items') \
  .load()
order_items.createOrReplaceTempView('order_items')

In [ ]:
# Total number of sales broken down by product in descending order
best_selling_item = spark.sql(
'SELECT oi.product_id as product_id, p.name as product_name, p.category as product_category, count(*) as num_of_orders \
FROM products as p \
JOIN order_items as oi \
ON p.id = oi.product_id \
GROUP BY 1,2,3 \
ORDER BY num_of_orders DESC')
best_selling_item.show()

+----------+--------------------+--------------------+-------------+
|product_id|        product_name|    product_category|num_of_orders|
+----------+--------------------+--------------------+-------------+
|     29035|Quiksilver Men's ...|         Accessories|           19|
|     20574|Levi's Men's 508 ...|               Jeans|           18|
|      5191|BLACK PANT GAUCHO...|      Pants & Capris|           18|
|     25912|Classic Cotton Li...|           Underwear|           18|
|     26420|Intymen Veil Boxe...|           Underwear|           17|
|     23852|Polo Ralph Lauren...|   Outerwear & Coats|           17|
|      4902|Not Your Daughter...|               Jeans|           17|
|     18972|Fred Perry Men's ...|            Sweaters|           17|
|     16519|Woolrich Men's Wo...|         Tops & Tees|           17|
|     21314|Wrangler Men's Re...|               Jeans|           17|
|     26338|Groovin' White V-...|           Underwear|           17|
|      2313|Allegra K Ladies ...|F

In [ ]:
# prompt: plot best_selling_item on a donut chart

import pandas as pd
import plotly.graph_objects as go

best_selling_item_pd = best_selling_item.toPandas()

# Aggregate the data for the donut chart
top_n = 10  # Display top N products
aggregated_data = best_selling_item_pd.groupby('product_category')['num_of_orders'].sum().reset_index()
aggregated_data = aggregated_data.sort_values('num_of_orders', ascending=False).head(top_n)

# Create the donut chart
fig = go.Figure(data=[go.Pie(labels=aggregated_data['product_category'],
                             values=aggregated_data['num_of_orders'],
                             hole=.3)])
fig.update_layout(title="Best Selling Product Categories (Donut Chart)")
fig.show()